# FLOPROS flood-protection standards — a guided tour

This notebook teaches the [`flopros`](../../reference/flopros/introduction.md) backend: how to fetch the **FLOPROS** global database of flood-protection standards (Scussolini et al., 2016) and explore it. The data is public (an NHESS-2016 supplement shapefile, CC-BY-3.0, no credentials).

FLOPROS is the **defended-vs-undefended correction**: each of ~4650 subnational polygons carries the flood protection in force as a **return period in years** — e.g. "defended against the 100-year flood". Clip a hazard map by the local standard and you separate protected from exposed areas.

## The request axes

A request selects a protection layer and, optionally, a unit or a region:

| Argument | Meaning | Values |
|---|---|---|
| `layer` | which protection layer(s) | `merged_riverine` (recommended), `modelled_riverine`, `design_*`/`policy_*` (min/max, riverine + coastal), or `None` for all |
| `country` | keep one unit | a `name`/`geonunit` (case-insensitive) |
| `lat_lim` / `lon_lim` | keep a region | a WGS84 bounding box |
| `geometry` | output shape | `True` -> `FeatureCollection`, `False` -> `DataFrame` |

A protection value of `0` means no standard is recorded for that layer in that unit.

## Setup

Imports and a single notebook-relative cache/output dir (`out/`, gitignored).

In [ ]:
import matplotlib.pyplot as plt

from earthlens.core import EarthLens

DATA = "out"

## Quickstart — map the merged protection standard

The shortest end-to-end call: the **merged riverine** layer for every subnational unit. The first call downloads the ~14 MB NHESS supplement zip and caches it under `out/`; every later call reuses it.

In [ ]:
fc = EarthLens(
    "flopros",
    layer="merged_riverine",
    path=DATA,
    cache_dir=DATA,
).download()

print(len(fc), "units |", list(fc.columns))

A choropleth of the `merged_riverine` column — the protection return period (years) per unit:

In [ ]:
ax = fc.plot(
    column="merged_riverine",
    cmap="YlGnBu",
    legend=True,
    legend_kwds={"label": "Protection standard (return period, yr)", "shrink": 0.6},
    figsize=(11, 5.5),
    edgecolor="grey",
    linewidth=0.1,
)
ax.set_title("FLOPROS merged riverine protection standard")
ax.set_axis_off()
plt.show()

The highest standards cluster in the wealthy, low-lying river basins of north-west Europe and East Asia — the pattern the ranked table below confirms.

In [ ]:
fc[["name", "geonunit", "merged_riverine"]].sort_values(
    "merged_riverine", ascending=False
).head(8).reset_index(drop=True)

## Compare the protection layers

Omit `layer=` to keep **all** FLOPROS layers at once — Modelled, Merged, and the Design / Policy min & max (riverine `_riverine` + coastal `_coastal`). This reuses the cached shapefile, so no new download.

In [ ]:
allfc = EarthLens("flopros", path=DATA, cache_dir=DATA).download()
layer_cols = [
    c for c in allfc.columns if c not in ("name", "geonunit", "type_en", "geometry")
]
layer_cols

The Design and Policy layers bracket the modelled estimate: for a single unit you can read the modelled standard against the design range of its actual defences.

In [ ]:
cols = [
    "name",
    "modelled_riverine",
    "merged_riverine",
    "design_min_riverine",
    "design_max_riverine",
]
allfc[allfc["merged_riverine"] > 0][cols].sort_values(
    "merged_riverine", ascending=False
).head(6).reset_index(drop=True)

## Filter by country or bounding box

Pass `country=` (matched on `name` or `geonunit`, case-insensitive) to keep one country's units, or `lat_lim`/`lon_lim` for a region. Both reuse the cache. `geometry=False` drops the geometry for a plain table.

In [ ]:
de = EarthLens(
    "flopros",
    country="Germany",
    geometry=False,
    path=DATA,
    cache_dir=DATA,
).download()
de[["name", "geonunit", "merged_riverine", "modelled_riverine"]].reset_index(drop=True)

## Takeaway

- One call shape — `layer` x (`country` / bbox) x `geometry` — covers the whole product.
- Values are **protection standards as return periods (years)**; `0` means no standard recorded.
- Use `merged_riverine` for a single representative number; the Design/Policy layers give the defended range.
- Join FLOPROS to a hazard map (e.g. from `aqueduct` or `gee`) to separate protected from exposed exposure.

See the [FLOPROS reference](../../reference/flopros/introduction.md) for the full layer list and licence.